# LangGraph — Integration with an LLM

## Outline
* Simple chatbot with StateGraph + LLM
* Adding the `add_messages` reducer
* Agent with Tool-Calling from scratch (without `create_agent`)
* Comparing raw StateGraph vs `create_agent`



In [1]:
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

## 1. Simple Chatbot with StateGraph

First, a reminder: in the previous notebook, State only contained `int` and `str`.  
Now State contains a **list of messages**.

### Problem: Reducer for messages

```
Without a reducer:
  Node A → {"messages": [msg1]}      # replaced
  Node B → {"messages": [msg2]}      # msg1 is lost!

With the add_messages reducer:
  Node A → {"messages": [msg1]}      # appended
  Node B → {"messages": [msg2]}      # msg1 + msg2 = both are preserved ✓
```

In [3]:
from typing import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_ollama import ChatOllama
import os


# ── 1. State with add_messages reducer ────────────────────
class ChatState(TypedDict):
    # add_messages: new messages are appended (not replaced)
    messages: Annotated[list[BaseMessage], add_messages]

# ── 2. LLM ──────────────────────────────────────────────
llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("BASE_URL"))
ollama = ChatOllama(model="qwen3.8:latest", temperature=0.7)

# ── 3. Node: a single function that calls the LLM ─────────────
def call_llm(state: ChatState) -> dict:
    """Node: sends State messages to the LLM and appends the response"""
    response = ollama.invoke(state["messages"])
    return {"messages": [response]}  # add_messages appends it

# ── 4. Build the graph ────────────────────────────────────────
builder = StateGraph(ChatState)
builder.add_node("llm", call_llm)
builder.add_edge(START, "llm")
builder.add_edge("llm", END)

memory = InMemorySaver()
chatbot = builder.compile(checkpointer=memory)

# ── 5. Test with memory ─────────────────────────────────────
config = {"configurable": {"thread_id": "test_1"}}

# First message
r1 = chatbot.invoke(
    {"messages": [HumanMessage(content="Hello! My name is Meisam and I am a developer.")]},
    config
)
print("Message 1:", r1["messages"][-1].content)

# Second message — the chatbot should remember the name
r2 = chatbot.invoke(
    {"messages": [HumanMessage(content="What is my name? and what is my profession?")]},
    config
)
print("\nMessage 2:", r2["messages"][-1].content)

Message 1: Hello, Meisam! Nice to meet you. 👋

Great to have a fellow developer in the conversation. Whether you're working on a new project, debugging a tricky issue, exploring a new language or framework, or just thinking through an architecture decision — I'm here to help.

What's on your mind today?

Message 2: Your name is **Meisam**, and your profession is a **developer**. You told me both of those in your first message. 😊

Anything else I can help you with?


## 2. Agent with Tool-Calling from Scratch

Now we will build the same Agent pattern that `create_agent` creates, ourselves with StateGraph.

```
ReAct Loop in LangGraph:

  START
    │
    ▼
  [agent]  ← LLM decides
    │
    ├─ tool_calls exist → [tools]  ← tools are executed
    │                              │
    │                              └──────────────────────┐
    │                                                     │
    │◄────────────────────────────────────────────────────┘
    │
    └─ no tool_calls → END
```

In [4]:
from typing import TypedDict, Annotated, Literal
from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_ollama import ChatOllama
import os

# ── 1. Tools ────────────────────────────────────────────
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b

@tool
def add(a: float, b: float) -> float:
    """Add two numbers"""
    return a + b

tools = [multiply, add]

# ── 2. LLM with tools bound ────────────────────────────
llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, 
                      api_key=os.getenv("OPENAI_API_KEY"), 
                      base_url=os.getenv("BASE_URL")
                      )
ollama = ChatOllama(model="qwen3.8:latest", temperature=0.7)
llm_with_tools = ollama.bind_tools(tools)

# ── 3. State ─────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

# ── 4. Nodes ────────────────────────────────────────────
def agent_node(state: AgentState) -> dict:
    """LLM decides whether to call a tool or answer"""
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

# Prebuilt ToolNode from LangGraph — automatically executes tools
tool_node = ToolNode(tools)

# ── 5. Router ────────────────────────────────────────────
def should_use_tool(state: AgentState) -> Literal["tools", "__end__"]:
    """If the LLM has a tool_call → tools, otherwise finish"""
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "__end__"

# ── 6. Build the graph ────────────────────────────────────────
builder = StateGraph(AgentState)

builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    should_use_tool,
    {"tools": "tools", "__end__": END}
)

builder.add_edge("tools", "agent")  # ← after the tool → back to the agent

react_agent = builder.compile()

# ── 7. Test ──────────────────────────────────────────────
print("=== Agent Test ===")
result = react_agent.invoke({
    "messages": [HumanMessage(content="What is (3 × 4) + 5?")]
})
print(result["messages"][-1].content)

=== Agent Test ===
(3 × 4) + 5 = **17**


### Viewing Execution Steps (Streaming)

In [5]:
print("=== Execution Steps ===")
for step in react_agent.stream(
    {"messages": [HumanMessage(content="What is (3 × 4) + 5?")]},
    stream_mode="updates"  # shows each update separately
):
    node_name = list(step.keys())[0]
    msgs = step[node_name]["messages"]
    last = msgs[-1]
    
    if node_name == "agent":
        if hasattr(last, "tool_calls") and last.tool_calls:
            for tc in last.tool_calls:
                print(f"  [agent → tool_call]  {tc['name']}({tc['args']})")
        else:
            print(f"  [agent → response]  {last.content}")
    elif node_name == "tools":
        print(f"  [tools → result]  {last.content}")

=== Execution Steps ===
  [agent → tool_call]  multiply({'a': 3, 'b': 4})
  [tools → result]  12.0
  [agent → tool_call]  add({'a': 12, 'b': 5})
  [tools → result]  17.0
  [agent → response]  (3 × 4) + 5 = **17**

- First: 3 × 4 = 12
- Then: 12 + 5 = **17**


## 3. Comparison: Raw StateGraph vs `create_agent`

```
create_agent (previous notebooks):
──────────────────────────────
  agent = create_agent(model=llm, tools=[...], system_prompt="...")
  # ← builds the same agent shown above, but you do not control the graph

StateGraph (this notebook):
──────────────────────────────
  builder = StateGraph(AgentState)
  builder.add_node("agent", agent_node)
  builder.add_node("tools", tool_node)
  builder.add_conditional_edges(...)
  # ← you control every step
```

| | `create_agent` | Raw StateGraph |
|---|---|---|
| **Speed** | Fast | Slower |
| **Control** | Limited | Full |
| **Custom State** | With AgentState | Any TypedDict |
| **Complex routing** | Limited | Flexible |
| **Use case** | Regular agent | Complex workflow |

> **General rule**: Start with `create_agent`. When you need more control, move to StateGraph.

## 4. Practical Example: Text Processing Graph

A pipeline that processes text:  
If it is Persian, it translates it, then writes a summary.

In [6]:
from typing import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

#llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
llm = ChatOllama(model="qwen3.8:latest", temperature=0.7)

# ── State  ─────────────────────────────────────
class TextState(TypedDict):
    text: str
    language: str       # "fa" or "en"
    english_text: str   # translation (if needed)
    summary: str

# ── Node: language detection ─────────────────────────────────────
def detect_language(state: TextState) -> dict:
    prompt = f"Answer only with 'fa' or 'en'. What is the language of this text:\n{state['text']}"
    lang = llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
    lang = "fa" if "fa" in lang else "en"
    print(f"  [detect_language]  language: {lang}")
    return {"language": lang}

# ── Node: translation ──────────────────────────────────────────
def translate_to_english(state: TextState) -> dict:
    prompt = f"Translate this text into English:\n{state['text']}"
    translated = llm.invoke([HumanMessage(content=prompt)]).content
    print(f"  [translate]  translated")
    return {"english_text": translated}

# ── Node: summary ──────────────────────────────────────────
def summarize(state: TextState) -> dict:
    source = state["english_text"] if state["english_text"] else state["text"]
    prompt = f"Summarize this text in one sentence:\n{source}"
    summary = llm.invoke([HumanMessage(content=prompt)]).content
    return {"summary": summary}

# ── Router ───────────────────────────────────────────────
def route_language(state: TextState) -> Literal["translate", "summarize"]:
    return "translate" if state["language"] == "fa" else "summarize"

# ── Build the graph ───────────────────────────────────────────
builder = StateGraph(TextState)
builder.add_node("detect", detect_language)
builder.add_node("translate", translate_to_english)
builder.add_node("summarize", summarize)

builder.add_edge(START, "detect")
builder.add_conditional_edges(
    "detect",
    route_language,
    {"translate": "translate", "summarize": "summarize"}
)
builder.add_edge("translate", "summarize")
builder.add_edge("summarize", END)

pipeline = builder.compile()

# ── Test with Persian text ─────────────────────────────────────
print("=== Persian Text ===")
res = pipeline.invoke({
    "text": "هوش مصنوعی در حال تغییر جهان است و در آینده‌ی بشریت نقش مهمی ایفا خواهد کرد.",
    "language": "", "english_text": "", "summary": ""
})
print(f"summary: {res['summary']}")

print("\n=== English Text ===")
res = pipeline.invoke({
    "text": "Artificial intelligence is transforming industries and creating new opportunities.",
    "language": "", "english_text": "", "summary": ""
})
print(f"summary: {res['summary']}")

print("\n=== French Text ===")
res = pipeline.invoke({
    "text": "L'intelligence artificielle transforme les industries et crée de nouvelles opportunités.",
    "language": "", "english_text": "", "summary": ""
})
print(f"summary: {res['summary']}")

=== Persian Text ===
  [detect_language]  language: fa
  [translate]  translated
summary: The text explains that artificial intelligence is reshaping the world and will be essential to humanity's future, with a brief note on the translation choices used to convey that message.

=== English Text ===
  [detect_language]  language: en
summary: AI is reshaping industries while opening up new opportunities.

=== French Text ===
  [detect_language]  language: en
summary: Artificial intelligence is reshaping industries while generating new opportunities.


```
Graph structure:

  START → [detect]
               │
               ├─ Persian → [translate] → [summarize] → END
               └─ English ───────────→ [summarize] → END
```

## Summary of the Entire LangGraph Course

```
Notebook 1 (08_09a) — Core concepts:
  ├── State (TypedDict)
  ├── Node (Python function)
  ├── Edge (direct)
  ├── Conditional Edge (conditional)
  ├── Reducer (Annotated + operator.add)
  ├── Loop
  └── Checkpointing

Notebook 2 (08_09b) — Integration with an LLM:
  ├── add_messages reducer
  ├── Chatbot with memory
  ├── ReAct Agent from scratch (ToolNode)
  ├── Streaming execution steps
  └── Multi-step pipeline

Next notebooks (08_09, 08_10, 08_11):
  ├── Custom State in create_agent
  ├── Multi-Agent
  └── Human-in-the-Loop
```